## Länkröta Stockholms stad
* Notebook [sthlm_65.ipynb](https://github.com/salgo60/SCB-Wikidata/blob/main/notebook/sthlm_65.ipynb) 
* [#65](https://github.com/salgo60/SCB-Wikidata/issues/65)
* result   
   * [html](https://salgo60.github.io/SCB-Wikidata/notebook/resultsStockholm/links_sthlm_v1_2026_01_25.html)
   * [csv](https://salgo60.github.io/SCB-Wikidata/notebook/resultsStockholm/links_sthlm_v1_2026_01_25.csv)

In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-02-21 09:27:40


In [2]:

SCRIPT_NAME = "sthlm_65.ipynb"
SCRIPT_URL = (
    "https://github.com/salgo60/SCB-Wikidata/"
    "blob/master/notebook/sthlm_65.ipynb"
) 


In [3]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [4]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []

    # --- language-specific sites ---
    for key, lang_block in data.items():
        if not key.isdigit():
            continue  # skip "count", "specials"

        lang_code = lang_block.get("code")
        lang_name = lang_block.get("name")

        for site in lang_block.get("site", []):
            rows.append({
                "lang_code": lang_code,
                "lang_name": lang_name,
                "project": site.get("project"),
                "url": site.get("url"),
                "dbname": site.get("dbname"),
                "site_name": site.get("sitename"),
                "closed": site.get("closed", False)
            })

    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [5]:
import requests
import pandas as pd


HEADERS = {
    "User-Agent": "salgo60-language-fetcher/2.0 (https://github.com/salgo60) salgo60@msn.com"
}


df_lang_fetch = fetch_sitematrix_df()
df_lang_fetch["closed"] = df_lang_fetch["closed"].fillna(False).astype(bool)

df_lang_wikipedia = df_lang_fetch[
    (df_lang_fetch["site_name"] == "Wikipedia") &
    (
        (df_lang_fetch["lang_name"].str.lower() != "special") |
        (df_lang_fetch["dbname"] == "wikidatawiki")
    )
]  
# Above also contains Wikidata 
# df_lang_fetch = fetch_sitematrix_df()
# df_lang_fetch["closed"] = df_lang_fetch["closed"].fillna(False).astype(bool)
#
# df_lang_wikipedia = df_lang_filtered = df_lang_fetch[
#    (df_lang_fetch["site_name"] == "Wikipedia") &
#    (df_lang_fetch["lang_name"].str.lower() != "special")]
# 
# 

#df_lang_wikipedia.to_csv("test.csv")
df_lang_wikipedia.info()

[DEBUG] Fetching sitematrix…
<class 'pandas.core.frame.DataFrame'>
Index: 186 entries, 0 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  186 non-null    object
 1   lang_name  185 non-null    object
 2   project    0 non-null      object
 3   url        186 non-null    object
 4   dbname     186 non-null    object
 5   site_name  186 non-null    object
 6   closed     186 non-null    bool  
dtypes: bool(1), object(6)
memory usage: 10.4+ KB


In [6]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [7]:
import os
import time
import random
import requests
import pandas as pd
from urllib.parse import urlparse
from tqdm.notebook import tqdm
file_path_domain = "sources/domains_sthlm.csv"
domains = read_domains(file_path_domain)
print(domains)


[DEBUG] Reading domains from: sources/domains_sthlm.csv
[DEBUG] Found 2 domains.
['stockholm', 'stockholm.se']


In [8]:
def resolve_api_base(lang):
    if lang == "special":
        # Wikidata (only valid "special" case in your pipeline)
        return "https://www.wikidata.org/w/api.php"

    return f"https://{lang}.wikipedia.org/w/api.php"


In [9]:
def request_timeout(lang):
    return 100 if lang == "special" else 10

In [10]:
# -----------------------------------------------------------
# Fetch exturlusage entries for one lang/domain
# -----------------------------------------------------------
def fetch_exturlusage(lang, domain):
    #base = f"https://{lang}.wikipedia.org/w/api.php"
    base = resolve_api_base(lang)
    params = {
        "action": "query",
        "format": "json",
        "list": "exturlusage",
        "euquery": domain,
        "eulimit": "max"
    }
    while True:
        #r = session.get(base, params=params, timeout=10)
        r = session.get(base, params=params, timeout=request_timeout(lang))        
        try:
            data = r.json()
        except ValueError:
            print(f"[WARN] {lang}: JSON decode failed")
            break

        for item in data.get("query", {}).get("exturlusage", []):
            yield {
                "lang": lang,
                "page_title": item.get("title"),
                "url": item.get("url"),
                "wiki_link": f"https://{lang}.wikipedia.org/wiki/{item.get('title').replace(' ', '_')}"
            }

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.3)

In [11]:

# -------------------------
# Session & helpers
# -------------------------
session = requests.Session()
session.headers.update({"User-Agent": "SCB-LinkAudit/1.0 salgo60@msn.com"})

# we need some filtering  

print("Antal Språk:",len(df_lang_wikipedia ))
results = []
for _, row in df_lang_wikipedia.iterrows():
    lang = row["lang_code"]
    url  = row["url"]
    lang_name = row["lang_name"]
    before = len(results)
    #print(lang, url, lang_name,domains)
    for entry in fetch_exturlusage(lang, domains):
        results.append(entry)     
    after = len(results) 
    links = after-before
    print(lang, url, lang_name," - ", links)
    


Antal Språk: 186
aa https://aa.wikipedia.org Qafár af  -  0
ace https://ace.wikipedia.org Acèh  -  0
af https://af.wikipedia.org Afrikaans  -  1
ak https://ak.wikipedia.org None  -  0
ami https://ami.wikipedia.org Pangcah  -  0
an https://an.wikipedia.org aragonés  -  1
ast https://ast.wikipedia.org asturianu  -  3
av https://av.wikipedia.org авар  -  0
avk https://avk.wikipedia.org Kotava  -  0
ay https://ay.wikipedia.org Aymar aru  -  0
bar https://bar.wikipedia.org Boarisch  -  3
bbc https://bbc.wikipedia.org Batak Toba  -  0
bcl https://bcl.wikipedia.org Bikol Central  -  0
bi https://bi.wikipedia.org Bislama  -  0
bm https://bm.wikipedia.org bamanankan  -  0
bo https://bo.wikipedia.org བོད་ཡིག  -  0
br https://br.wikipedia.org brezhoneg  -  1
bs https://bs.wikipedia.org bosanski  -  4
btm https://btm.wikipedia.org Batak Mandailing  -  0
bug https://bug.wikipedia.org Basa Ugi  -  0
bxr https://bxr.wikipedia.org буряад  -  1
cbk-zam https://cbk-zam.wikipedia.org Chavacano de Zamboan

In [12]:
df_sthlm = pd.DataFrame(results)
df_sthlm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11824 entries, 0 to 11823
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        11824 non-null  object
 1   page_title  11824 non-null  object
 2   url         11824 non-null  object
 3   wiki_link   11824 non-null  object
dtypes: object(4)
memory usage: 369.6+ KB


In [13]:
import pandas as pd

# --- Stats ---
total_links = len(df_sthlm)
total_unique_links = df_sthlm['url'].nunique()
num_languages = df_sthlm['lang'].nunique()
langs_sorted = df_sthlm['lang'].value_counts()

print("Total links:", total_links)
print("Total unique links:", total_unique_links)
print("Number of languages:", num_languages)
print("\nLanguages with most links:")
print(langs_sorted.to_string())


Total links: 11824
Total unique links: 6706
Number of languages: 56

Languages with most links:
lang
sv          8367
special     1033
en           803
no           482
pl           239
es           166
fi           165
de           134
da           105
it            65
zh            52
nl            25
vi            23
ja            19
nn            17
id            16
sq            13
eu            11
gl             9
ro             7
simple         6
su             5
ms             4
tt             4
fo             4
bs             4
bar            3
so             3
sco            3
diq            3
ast            3
new            2
ha             2
szl            2
gv             2
lld            2
pcd            2
smn            1
zu             1
stq            1
sw             1
tet            1
af             1
se             1
roa-tara       1
nds            1
an             1
li             1
kv             1
kaa            1
ilo            1
ig             1
eml            

In [14]:
# --- Stats ---
total_links = len(df_sthlm)
total_unique_links = df_sthlm["url"].nunique()
langs_with_hits = sorted(df_sthlm["lang"].unique())

num_languages_found = len(langs_with_hits)
num_languages_checked = len(df_lang_wikipedia)        # alla språk som genomsöktes
num_languages_found = df_sthlm['lang'].nunique()


In [15]:
num_languages_checked 

186

In [16]:
import requests
from requests.exceptions import RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd
import time
import threading

# ==========================
# Konfiguration
# ==========================

OUTPUT = "checkpoint_links_sthlm.parquet"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 15
CHECKPOINT_EVERY = 400

ROOT_CANONICAL = "https://start.stockholm"

SOFT_404_PHRASES = [
    "Sidan hittades inte",
    "sidan tagits bort",
    "felaktig adress",
    "kontakta registrator",
    "sidan finns inte",
    "den här sidan kan inte visas",
    "Hoppsan! Vi kunde tyvärr inte hitta sidan"
]

USER_AGENT = "LinkChecker/1.0 (research; salgo60@msn.com)"

# ==========================
# Session (thread-safe)
# ==========================

thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        thread_local.session = s
    return thread_local.session


# ==========================
# Hjälpfunktioner
# ==========================

def norm(u: str) -> str:
    return u.rstrip("/").lower()


def looks_like_soft_404(response) -> bool:
    text = (response.text or "").lower()

    for phrase in SOFT_404_PHRASES:
        if phrase in text:
            return True

    if "<title>" in text:
        title = text.split("<title>", 1)[1].split("</title>", 1)[0]
        if "404" in title.lower():
            return True
        if "sidan finns inte" in title.lower():
            return True

    return False



# ==========================
# URL-check
# ==========================

def check_url(url: str) -> dict:
    session = get_session()

    try:
        # 1. HEAD först
        r = session.head(
            url,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {
            "url": url,
            "status": "error",
            "reason": str(e),
        }

    final_url = r.url
    status_code = r.status_code

    # Hard error
    if status_code >= 400:
        return {
            "url": url,
            "status": "dead",
            "reason": f"HTTP {status_code}",
            "final_url": final_url,
        }

    # Redirect till root
    if norm(final_url) == norm(ROOT_CANONICAL) and norm(url) != norm(final_url):
        return {
            "url": url,
            "status": "dead",
            "reason": "redirect_to_root",
            "final_url": final_url,
        }

    # 2. GET endast vid behov
    try:
        r = session.get(
            final_url,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {
            "url": url,
            "status": "error",
            "reason": str(e),
            "final_url": final_url,
        }

    if looks_like_soft_404(r):
        return {
            "url": url,
            "status": "dead",
            "reason": "soft_404",
            "final_url": final_url,
        }

    return {
        "url": url,
        "status": "ok",
        "final_url": final_url,
    }


# ==========================
# Internet Archive
# ==========================

def check_internet_archive(url: str) -> str | None:
    session = get_session()
    api = "https://archive.org/wayback/available"

    try:
        r = session.get(
            api,
            params={"url": url},
            timeout=10,
        )
        data = r.json()
    except Exception:
        return None

    snap = data.get("archived_snapshots", {}).get("closest")
    if snap and snap.get("available"):
        return snap.get("url")

    return None


# ==========================
# Worker
# ==========================

def process_url(url: str) -> dict:
    result = check_url(url)

    if result["status"] == "dead":
        ia_url = check_internet_archive(url)
        result["ia_url"] = ia_url
        result["ia_status"] = "available" if ia_url else "missing"
    else:
        result["ia_url"] = None
        result["ia_status"] = "skipped"

    return result


# ==========================
# Main
# ==========================

def run(df_sthlm: pd.DataFrame) -> pd.DataFrame:
    urls = [
        u for u in df_sthlm["url"].dropna().astype(str).unique()
    ]

    results = []
    start = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_url, u): u for u in urls}

        for i, future in enumerate(
            tqdm(as_completed(futures), total=len(futures), unit="link"),
            start=1
        ):
            result = future.result()
            results.append(result)

            if i % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results).to_parquet(OUTPUT)

    df = pd.DataFrame(results)
    df.to_parquet(OUTPUT)

    elapsed = time.time() - start
    print(f"Klar på {elapsed/3600:.1f} timmar")

    return df


In [17]:
df_sthlm


,lang,page_title,url,wiki_link
0,af,Bespreking:Stockholm,http://www.stockholm.se/-/English/Politics-and...,https://af.wikipedia.org/wiki/Bespreking:Stock...
1,an,Estocolmo,http://www.stockholm.se/,https://an.wikipedia.org/wiki/Estocolmo
2,ast,Concordia Selander,http://hittagraven.stockholm.se/Sok/?&cemetery...,https://ast.wikipedia.org/wiki/Concordia_Selander
3,ast,Naemi Briese,http://hittagraven.stockholm.se/sv/Galarvarvsk...,https://ast.wikipedia.org/wiki/Naemi_Briese
4,ast,Sophie Cysch,https://stockholmskallan.stockholm.se/PostFile...,https://ast.wikipedia.org/wiki/Sophie_Cysch
...,...,...,...,...
11819,special,Q133884437,https://stockholmskallan.stockholm.se/post/33550,https://special.wikipedia.org/wiki/Q133884437
11820,special,Q10680462,https://meetingspublic.stockholm.se/committees...,https://special.wikipedia.org/wiki/Q10680462
11821,special,Q10680557,https://meetingspublic.stockholm.se/committees...,https://special.wikipedia.org/wiki/Q10680557
11822,special,Q19976408,https://etjanster.stockholm.se/Hittagraven/hom...,https://special.wikipedia.org/wiki/Q19976408


In [18]:
df_result = run(df_sthlm)

100%|█████████████████████████████████████| 6706/6706 [36:32<00:00,  3.06link/s]

Klar på 0.6 timmar


In [19]:
df_sthlm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11824 entries, 0 to 11823
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        11824 non-null  object
 1   page_title  11824 non-null  object
 2   url         11824 non-null  object
 3   wiki_link   11824 non-null  object
dtypes: object(4)
memory usage: 369.6+ KB


In [20]:

print("len(results):", len(results))

print("df_result.shape:", df_result.shape)
print(df_result.head(2))
print(df_result.tail(2))


len(results): 11824
df_result.shape: (6706, 6)
                                                 url status  \
0  http://www.opac.ssb.stockholm.se/pls/bookit/pk...  error   
1  http://www.usk.stockholm.se/internet/omrfakta/...  error   

                                              reason ia_url ia_status  \
0  HTTPConnectionPool(host='www.opac.ssb.stockhol...   None   skipped   
1  HTTPConnectionPool(host='www.usk.stockholm.se'...   None   skipped   

  final_url  
0       NaN  
1       NaN  
                                                    url status reason ia_url  \
6704  https://edokmeetings.stockholm.se/welcome-sv/n...     ok    NaN   None   
6705  https://miljobarometern.stockholm.se/content/d...     ok    NaN   None   

     ia_status                                          final_url  
6704   skipped  https://meetingspublic.stockholm.se:443/404?as...  
6705   skipped  https://miljobarometern.stockholm.se/content/d...  


In [21]:
print("len(results):", len(results))
print("first:", results[0])
print("last:", results[-1])

len(results): 11824
first: {'lang': 'af', 'page_title': 'Bespreking:Stockholm', 'url': 'http://www.stockholm.se/-/English/Politics-and-organisation/Organisation-/Council-of-Mayors-/#Sten-Nordin-m', 'wiki_link': 'https://af.wikipedia.org/wiki/Bespreking:Stockholm'}
last: {'lang': 'special', 'page_title': 'Q137668588', 'url': 'https://stockholmskallan.stockholm.se/post/11741', 'wiki_link': 'https://special.wikipedia.org/wiki/Q137668588'}


In [22]:
results[:3]

[{'lang': 'af',
  'page_title': 'Bespreking:Stockholm',
  'url': 'http://www.stockholm.se/-/English/Politics-and-organisation/Organisation-/Council-of-Mayors-/#Sten-Nordin-m',
  'wiki_link': 'https://af.wikipedia.org/wiki/Bespreking:Stockholm'},
 {'lang': 'an',
  'page_title': 'Estocolmo',
  'url': 'http://www.stockholm.se/',
  'wiki_link': 'https://an.wikipedia.org/wiki/Estocolmo'},
 {'lang': 'ast',
  'page_title': 'Concordia Selander',
  'url': 'http://hittagraven.stockholm.se/Sok/?&cemetery=22&lastName=selander&dateOfBurial=1935',
  'wiki_link': 'https://ast.wikipedia.org/wiki/Concordia_Selander'}]

In [23]:
df_results = (
    pd.DataFrame(results)
    .drop_duplicates(subset="url", keep="last")
)
df_sthlm = df_sthlm.merge(
    df_result,
    on="url",
    how="left",
) 

In [24]:
df_sthlm.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11824 entries, 0 to 11823
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        11824 non-null  object
 1   page_title  11824 non-null  object
 2   url         11824 non-null  object
 3   wiki_link   11824 non-null  object
 4   status      11824 non-null  object
 5   reason      7483 non-null   object
 6   ia_url      1645 non-null   object
 7   ia_status   11824 non-null  object
 8   final_url   8581 non-null   object
dtypes: object(9)
memory usage: 831.5+ KB


In [25]:
status_counts = df_sthlm["status"].value_counts()

num_ok = status_counts.get("ok", 0)
num_dead = status_counts.get("dead", 0)
num_error = status_counts.get("error", 0)
num_total = len(status_counts)
print( "Ok ",num_ok) 
print( "Dead ",num_dead)
print( "Error ",num_error ) 
print( "Total ",num_total )

Ok  4341
Dead  4240
Error  3243
Total  3


In [26]:
df_sthlm["reason"].value_counts()

reason
HTTP 404                                                                                                                                                                                                                                                                                                                                                                                                   1986
HTTP 405                                                                                                                                                                                                                                                                                                                                                                                                    926
HTTP 400                                                                                                                                                                                         

In [27]:
from datetime import date
import os

# Sätt datum
today = date.today().strftime("%Y_%m_%d")

# Se till att katalogen finns
os.makedirs("resultsStockholm", exist_ok=True)

# Bygg filnamn
outfile = f"resultsStockholm/links_sthlm_v1_{today}.csv"

# Exportera
df_sthlm.to_csv(outfile, index=False, encoding="utf-8")

print(f"[OK] Exported {len(df_sthlm)} rows to {outfile}")


[OK] Exported 11824 rows to resultsStockholm/links_sthlm_v1_2026_02_21.csv


In [28]:
lang_stats = (
    df_sthlm
    .groupby("lang")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        archived_links=("ia_status", lambda s: (s == "available").sum()),
    )
    .reset_index()
)

lang_stats["broken_pct"] = (
    100 * lang_stats["broken_links"] / lang_stats["total_links"]
).round(1)

lang_stats["broken_lost"] = (
    lang_stats["broken_links"] - lang_stats["archived_links"])

top10_langs = (
    lang_stats
    .sort_values("total_links", ascending=False)
    .head(10)
)
top10_langs[
    [
        "lang",
        "total_links",
        "broken_links",
        "broken_pct",
        "archived_links",
        "broken_lost",
    ]
]

,lang,total_links,broken_links,broken_pct,archived_links,broken_lost
48,sv,8367,2826,33.8,816,2010
44,special,1033,317,30.7,222,95
11,en,803,238,29.6,134,104
34,no,482,345,71.6,152,193
36,pl,239,192,80.3,96,96
12,es,166,33,19.9,19,14
14,fi,165,59,35.8,25,34
8,de,134,42,31.3,31,11
7,da,105,38,36.2,28,10
22,it,65,30,46.2,26,4


In [29]:
from urllib.parse import urlparse

df = df_sthlm.copy()

df["domain"] = df["url"].apply(
    lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
)
domain_stats = (
    df
    .groupby("domain")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        error_links=("status", lambda s: (s == "error").sum()),
    )
    .reset_index()
)
domain_stats["broken_pct"] = (
    100 * domain_stats["broken_links"] / domain_stats["total_links"]
).round(1)

domain_stats["error_pct"] = (
    100 * domain_stats["error_links"] / domain_stats["total_links"]
).round(1)


In [30]:
domain_stats 

,domain,total_links,broken_links,error_links,broken_pct,error_pct
0,150.stockholm.se,2,2,0,100.0,0.0
1,abrahamsbergsskolan.stockholm.se,1,0,1,0.0,100.0
2,adolffredriksmusikklasser.stockholm.se,14,0,14,0.0,100.0
3,akallagrundskola.stockholm.se,3,0,0,0.0,0.0
4,alstensskolan.stockholm.se,4,2,0,50.0,0.0
...,...,...,...,...,...,...
293,www.vallingby.stockholm.se,1,0,1,0.0,100.0
294,www.vinstagrundskola.stockholm.se,1,0,1,0.0,100.0
295,www2.ssa.stockholm.se,88,0,88,0.0,100.0
296,www2.stockholm.se,6,0,6,0.0,100.0


In [31]:
status_counts = df["status"].value_counts()

num_ok = int(status_counts.get("ok", 0))
num_dead = int(status_counts.get("dead", 0))
num_error = int(status_counts.get("error", 0))
num_total = len(df)

pct_ok = round(100 * num_ok / num_total, 1)
pct_dead = round(100 * num_dead / num_total, 1)
pct_error = round(100 * num_error / num_total, 1)

# Broken links: archived vs lost
num_dead_archived = df[
    (df["status"] == "dead") & (df["ia_status"] == "available")
].shape[0]

num_dead_lost = num_dead - num_dead_archived


In [32]:
top_domains = (
    domain_stats[domain_stats["total_links"] >= 5]
    .sort_values("total_links", ascending=False)
    .head(10)
)
domain_stats_html = "<ul>"
for _, r in top_domains.iterrows():
    domain_stats_html += (
        f"<li><strong>{r['domain']}</strong>: "
        f"{r['broken_links']} / {r['total_links']} broken "
        f"({r['broken_pct']}%)</li>"
    )
domain_stats_html += "</ul>"


In [33]:
from pathlib import Path
from datetime import date, datetime
from urllib.parse import quote
import pandas as pd



def is_qid(title):
    return isinstance(title, str) and title.startswith("Q") and title[1:].isdigit()

def save_sortable_html_df_sthlm(
    df,
    out_dir="resultsStockholm",
    domains=None,
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/65",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    out_path = out_dir / f"links_sthlm_v1_{today}.html"
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

     # --- Förbered data ---
    df = df.copy()  
    status_counts = df["status"].value_counts()
    df["is_wikidata"] = df["page_title"].apply(is_qid)
    
    # Normalize lang
    df.loc[df["is_wikidata"], "lang"] = "wikidata"
    df["lang"] = df["lang"].replace({"special": None})

    num_ok = int(status_counts.get("ok", 0))
    num_dead = int(status_counts.get("dead", 0))
    num_error = int(status_counts.get("error", 0))
    num_total = len(df)
    
    pct_ok = round(100 * num_ok / num_total, 1)
    pct_dead = round(100 * num_dead / num_total, 1)
    pct_error = round(100 * num_error / num_total, 1)
    
    # Broken links: archived vs lost
    num_dead_archived = df[
        (df["status"] == "dead") & (df["ia_status"] == "available")
    ].shape[0]
    
    num_dead_lost = num_dead - num_dead_archived

    domains = domains or []
    from urllib.parse import urlparse
    
    df["domain"] = df["url"].apply(
        lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
    )

    domain_stats = (
        df
        .groupby("domain")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
        )
        .reset_index()
    )
    
    domain_stats["broken_pct"] = (
        100 * domain_stats["broken_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["error_pct"] = (
        100 * domain_stats["error_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["problem_pct"] = (
        100 * (domain_stats["broken_links"] + domain_stats["error_links"])
        / domain_stats["total_links"]
    ).round(1)
    

    domain_stats_html = "<ul>"
    for _, r in top_domains.iterrows():
        domain_stats_html += (
            f"<li><strong>{r['domain']}</strong>: "
            f"{r['broken_links']} / {r['total_links']} broken "
            f"({r['broken_pct']}%)</li>"
        )
    domain_stats_html += "</ul>"

    domain_table_html = (
        domain_stats
        .head(100)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )
    lang_stats = (
    df
        .groupby("lang")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
            broken_archived=("ia_status", lambda s: (s == "available").sum()),
        )
        .reset_index()
    )
    
    lang_stats["broken_lost"] = (
        lang_stats["broken_links"] - lang_stats["broken_archived"]
    )
    
    lang_stats["broken_pct"] = (
        100 * lang_stats["broken_links"] / lang_stats["total_links"]
    ).round(1)
    
    lang_stats["problem_pct"] = (
        100 * (lang_stats["broken_links"] + lang_stats["error_links"])
        / lang_stats["total_links"]
    ).round(1)

    lang_stats = lang_stats.sort_values(
        "broken_links",
        ascending=False
    )
 
    lang_stats_display = lang_stats[
        [
            "lang",
            "total_links",
            "broken_links",
            "broken_archived",
            "broken_lost",
            "broken_pct",
            "problem_pct",
        ]
    ]

    lang_table_html = (
        lang_stats_display
        .head(15)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )

    STATUS_ICON = {
        "ok":    ("fa-circle-check", "#2e7d32", "OK"),
        "dead":  ("fa-circle-xmark", "#c62828", "Broken link"),
        "error": ("fa-triangle-exclamation", "#ef6c00", "Request error"),
    }
    
    if "status" in df.columns:
        def render_status(r):
            icon, color, label = STATUS_ICON.get(
                r["status"], ("fa-question-circle", "#757575", "Unknown")
            )
            reason = r.get("reason", "")
            return (
                f'<span class="status-icon" '
                f'data-status="{r["status"]}" '
                f'title="{label}: {reason}" '
                f'style="color:{color}; font-size:14px; cursor:pointer;">'
                f'<i class="fa-solid {icon}"></i>'
                f'</span>'
            )
        df.insert(
            0,
            "status_icon",
            df.apply(render_status, axis=1)
        )

        def render_ia_icon(r):
            if r.get("ia_status") == "available" and r.get("ia_url"):
                return (
                    f'<a href="{r["ia_url"]}" target="_blank" '
                    f'title="Archived copy (Internet Archive)">'
                    f'<i class="fa-solid fa-box-archive" '
                    f'style="color:#1565c0;"></i>'
                    f'</a>'
                )
            return ""
        
        df.insert(
            1,
            "archive",
            df.apply(render_ia_icon, axis=1)
        )


    # Wikipedia: ikon + titel (byggd från lang + page_title)
    if {"lang", "page_title"}.issubset(df.columns):
    
        def render_page_link(r):
            title = r["page_title"]
    
            # --- Wikidata item ---
            if r.get("is_wikidata"):
                label = r.get("wd_label_sv")
                link_text = label if pd.notna(label) else title
    
                return (
                    f'<a href="https://www.wikidata.org/wiki/{title}" '
                    f'target="_blank" title="Wikidata">'
                    f'<i class="fa-solid fa-database" style="margin-right:6px;"></i>'
                    f'{link_text}</a>'
                )
    
            # --- Wikipedia article ---
            lang = r.get("lang")
            if pd.notna(lang) and pd.notna(title):
                return (
                    f'<a href="https://{lang}.wikipedia.org/wiki/{quote(str(title))}" '
                    f'target="_blank" title="Wikipedia ({lang})">'
                    f'<i class="fa-brands fa-wikipedia-w" style="margin-right:6px;"></i>'
                    f'{title}</a>'
                )
    
            return title

        df["page_title"] = df.apply(render_page_link, axis=1)

    # Externa länkar
    for col in ["Wikipedia-länk", "Extern länk", "url"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if pd.notna(x) else ""
            )

    # --- HTML-tabell ---
    html_table = df.to_html(
        classes="pivot",
        border=0,
        escape=False,  # krävs för HTML-länkar
        index=False,
    )

    # --- CSS ---
    css = """
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 20px;
      }
      table.pivot {
        border-collapse: collapse;
        width: 100%;
        font-size: 12px;
      }
      table.pivot th, table.pivot td {
        border: 1px solid #999;
        padding: 6px 8px;
        text-align: left;
        vertical-align: top;
        white-space: normal;
      }
      table.pivot th {
        cursor: pointer;
        background: #f2f2f2;
      }
      table.pivot th:hover {
        background: #e2e2e2;
      }
      table.pivot thead th {
        position: sticky;
        top: 0;
        background: #f2f2f2;
        z-index: 2;
      }
      table.pivot th::after {
        content: "";
        float: right;
        opacity: 0.4;
      }
      table.pivot th.sorted-asc::after {
        content: " ▲";
      }
      table.pivot th.sorted-desc::after {
        content: " ▼";
      }
      /* Row coloring by status */
      table.pivot tr[data-status="dead"] {
         background-color: #fdecea;  /* light red */
      }
      table.pivot tr[data-status="dead"] td:nth-child(2) i {
          color: #1565c0;
        }

      table.pivot tr[data-status="error"] {
          background-color: #fff4e5;  /* light orange */
      }

      table.pivot td a {
        color: #0645ad;
        text-decoration: none;
      }
      table.pivot td a:hover {
        text-decoration: underline;
      }
      .meta {
        background: #f8f8f8;
        border: 1px solid #ccc;
        padding: 12px;
        margin-bottom: 20px;
        font-size: 13px;
      }
      .meta h2 {
        margin-top: 0;
      }
    </style>
    """

    # --- JavaScript (sortering) ---
    js = """
    <script>
    document.addEventListener('DOMContentLoaded', () => {
        // Propagate status from first cell to row
        document.querySelectorAll("table.pivot tbody tr").forEach(row => {
            const statusCell = row.querySelector(".status-icon");
            if (statusCell) {
                row.dataset.status = statusCell.dataset.status;
            }
        });
        let showOnlyBroken = false;

        document.querySelectorAll(".status-icon").forEach(icon => {
            icon.addEventListener("click", event => {
                event.stopPropagation(); // prevent column sort
                showOnlyBroken = !showOnlyBroken;
        
                document.querySelectorAll("table.pivot tbody tr").forEach(row => {
                    if (showOnlyBroken) {
                        row.style.display =
                            row.dataset.status === "dead" ? "" : "none";
                    } else {
                        row.style.display = "";
                    }
                });
            });
        });

        document.querySelectorAll("table.pivot th").forEach((header, colIndex) => {
            header.addEventListener("click", () => {
                const table = header.closest("table");
                const tbody = table.querySelector("tbody");
                const rows = Array.from(tbody.querySelectorAll("tr"));
                const asc = !header.classList.contains("sorted-asc");

                rows.sort((a, b) => {
                    const A = a.children[colIndex].innerText.trim();
                    const B = b.children[colIndex].innerText.trim();
                    const numA = parseFloat(A.replace(",", "."));
                    const numB = parseFloat(B.replace(",", "."));
                    if (!isNaN(numA) && !isNaN(numB)) {
                        return asc ? numA - numB : numB - numA;
                    }
                    return asc ? A.localeCompare(B) : B.localeCompare(A);
                });

                table.querySelectorAll("th").forEach(th =>
                    th.classList.remove("sorted-asc", "sorted-desc")
                );
                header.classList.add(asc ? "sorted-asc" : "sorted-desc");
                rows.forEach(row => tbody.appendChild(row));
            });
        });
    });
    </script>
    """
    status_counts = df["status"].value_counts()
    
    num_ok = status_counts.get("ok", 0)
    num_dead = status_counts.get("dead", 0)
    num_error = status_counts.get("error", 0)
    num_total = len(df)

    # --- Metadata ---
    meta_html = f"""
    <div class="meta">
      <h2>Summary</h2>
    
      <p><strong>Rerun:</strong> {rerun_ts}</p>
      <p><strong>Script:</strong>
         <a href="{SCRIPT_URL}" target="_blank">{SCRIPT_NAME}</a>
      </p>
    
      <p>
        <strong>Links checked:</strong> {num_total}<br>
        <strong style="color:#2e7d32;">OK:</strong> {num_ok} ({pct_ok}%)<br>
        <strong style="color:#c62828;">Broken:</strong> {num_dead} ({pct_dead}%)<br>
        &nbsp;&nbsp;↳ Archived: {num_dead_archived}<br>
        &nbsp;&nbsp;↳ Lost: {num_dead_lost}<br>
        <strong style="color:#ef6c00;">Errors:</strong> {num_error} ({pct_error}%)
      </p>
      <p><strong>Issue:</strong>
         <a href="{issue_url}" target="_blank">{issue_url.split("/")[-1]}</a>
      </p>
    
      <p><strong>Datakällor:</strong><br>
         Wikidata<br>
         MediaWiki API – exturlusage
      </p>
    
      <h2>Domains with broken links</h2>
      <p>Top domains ranked by broken-link impact.</p>
      {domain_table_html}
    </div>
    """


    # --- Slutlig HTML ---
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>stockholm links in Wikipedia</title>
      <link rel="stylesheet"
            href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.1/css/all.min.css">
      {css}
    </head>
    <body>
      <h1>Wikipedia → Stockholms stad v1</h1>
      {meta_html}
      <p>Sorterbar tabell. Klicka på kolumnrubriker för sortering.</p>
      {html_table}
      {js}
      <h2>Broken-link summary by Wikipedia language</h2>
     <p>
       Languages ranked by broken-link impact (broken + error links).
     </p>
    {lang_table_html}
    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ HTML skapad: {out_path}")


In [34]:

save_sortable_html_df_sthlm(df_sthlm)


✅ HTML skapad: resultsStockholm/links_sthlm_v1_2026_02_21.html


In [35]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2026-02-21 10:05:36
Total time elapsed: 37 minutes 56.49 seconds
